In [13]:
#!/home/damoncht/.conda/envs/ml/bin/python
import numpy as np
from tqdm import tqdm 
from multiprocessing import Pool, cpu_count
import time

def simNoise(sqrtSn=1, Tsft=7200, size=(256,128), ndet=2, norm=True):
    """
    Generates Gaussian noise in the frequency domain with given parameters.

    Parameters:
        Tsft (float): Total time duration for Fourier transform.
        size (tuple): Shape of each dataset in the form (num_bins, num_segments).
        ndet (int): Number of detector.

    Returns:
        np.ndarray: Generated noise data with shape (num_segments, num_bins, 2*ndet).
    """
    num_bins, num_segments = size
    # sigma in time domain = 1 and therefor in complex domain (real + imag), the variance is half of the original one 
    sigma = 0.5 * sqrtSn * np.sqrt(Tsft)  # Variance for real and imaginary parts 

    dataset = np.empty((num_bins, num_segments, 2*ndet))
    for i in range(ndet):  # Two time series
        dataset[...,2*i] = np.random.normal(0, sigma, size)
        dataset[...,2*i+1] = np.random.normal(0, sigma, size)
        
    if norm:
        return normalize(dataset)
    else:
        return dataset
    

def normalize(data):

    """
    Normalizes each channel of the image independently.
    
    Parameters:
        image (numpy.ndarray): Input image of shape (m, n, c).
    
    Returns:
        numpy.ndarray: Normalized image of the same shape.
    """
    normalized_image = (data - np.min(data)) / (np.max(data) - np.min(data))
    normalized_image = normalized_image * 2 - 1  # Scale to [-1, 1]

    #xmax = np.max(data, axis=(0, 1), keepdims=True)
    #xmin = np.min(data, axis=(0, 1), keepdims=True)    
    #normalized_image = (data - xmin) / (xmax-xmin) * 2 - 1
    return normalized_image

In [14]:
filename = '/scratch/kriles_root/kriles0/damoncht/unet_dyn_fastnoise/data/validation/500Hz_H1L1_D0-35_256x128_7200s_4c_traindata_n200_seed0.npz'
data = np.load(filename, allow_pickle=True)


In [17]:
for level in noise_levels_to_sample:
    if str(level) in noisy_data_dict:
        print(level)

In [2]:
t0 = time.time()
for seed in tqdm(range(300, 301)):
    dataset = np.empty((1000, 256,  128, 4))
    np.random.seed(seed)
    for i in range(1000):
         dataset[i] = simNoise(sqrtSn=1, Tsft=7200, size=(256,128), ndet=2, norm=False)

    np.savez("./data/pure_noise/{}_purenoise_n{}_seed{}.npz".format('H1L1', 1000, seed), dataset=dataset)

print("Time used ={} s".format(time.time()-t0))
print("Dataset saved successfully.")


100%|██████████| 1/1 [00:07<00:00,  7.12s/it]

Time used =7.134610414505005 s
Dataset saved successfully.
